In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install librosa torch seaborn --quiet
!pip install xgboost catboost --quiet
!pip install lightgbm --quiet
!pip install wandb --quiet
!pip install transformers datasets --quiet

import wandb
import numpy as np
import pandas as pd
import os
import librosa
import librosa.display
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,confusion_matrix,classification_report,accuracy_score

from transformers import AutoFeatureExtractor, HubertForSequenceClassification
import torch


In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

wandb_api = user_secrets.get_secret("Weights_and_biases_api")

os.environ["WANDB_API_KEY"] = wandb_api

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

GENRES_PATH = f"{DATA_PATH}/genres_stems"

ESC_PATH = f"{DATA_PATH}/ESC-50-master/audio"

GENRES = sorted(os.listdir(GENRES_PATH))

STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

print("Genres:", GENRES)

print("Total genres:", len(GENRES))

In [ ]:
noise_files = [
    f"{ESC_PATH}/{f}"
    for f in os.listdir(ESC_PATH)
    if f.endswith(".wav")
]

print("Noise files:",len(noise_files))

In [ ]:
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")

print("Test samples:", len(test_df))

test_df.head()

## Dataset Exploration (EDA)

In [ ]:
# genre = "rock"
# song_folder = os.listdir(f"{GENRES_PATH}/{genre}")[0]
# file_path = f"{GENRES_PATH}/{genre}/{song_folder}/vocals.wav"
# audio, sr = librosa.load(file_path, sr=None)
# print(sr, len(audio)/sr)


In [ ]:
# plt.figure(figsize=(12,4))
# librosa.display.waveshow(audio, sr=sr)
# plt.show()


In [ ]:
# spec = librosa.feature.melspectrogram(y=audio, sr=sr)
# spec_db = librosa.power_to_db(spec, ref=np.max)

# plt.figure(figsize=(12,4))
# librosa.display.specshow(spec_db, sr=sr, x_axis='time', y_axis='mel')
# plt.colorbar()
# plt.show()


In [ ]:
# lengths = []

# for genre in os.listdir(GENRES_PATH):
#     songs = os.listdir(f"{GENRES_PATH}/{genre}")
    
#     for song in songs:
#         path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
#         if os.path.exists(path):
#             audio, sr = librosa.load(path, sr=None)
#             lengths.append(len(audio)/sr)

# plt.hist(lengths, bins=20)
# plt.title("Audio Length Distribution")
# plt.xlabel("Seconds")
# plt.show()

# print("Mean length:", np.mean(lengths))


In [ ]:
# genre_counts = {}

# for genre in os.listdir(GENRES_PATH):
#     genre_counts[genre] = len(os.listdir(f"{GENRES_PATH}/{genre}"))

# pd.Series(genre_counts).sort_values().plot(kind="bar", figsize=(10,4))
# plt.title("Number of Songs per Genre")
# plt.show()


In [ ]:
# sample_rates = []

# for genre in os.listdir(GENRES_PATH):
#     songs = os.listdir(f"{GENRES_PATH}/{genre}")[:10]
#     for song in songs:
#         path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
#         if os.path.exists(path):
#             _, sr = librosa.load(path, sr=None)
#             sample_rates.append(sr)

# pd.Series(sample_rates).value_counts()


In [ ]:
# def plot_genre_mfcc(genre):
#     song = os.listdir(f"{GENRES_PATH}/{genre}")[0]
#     path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
#     audio, sr = librosa.load(path, sr=None)
#     mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    
#     plt.figure(figsize=(8,3))
#     librosa.display.specshow(mfcc, x_axis='time')
#     plt.title(genre)
#     plt.show()

# plot_genre_mfcc("rock")
# plot_genre_mfcc("classical")
# plot_genre_mfcc("hiphop")


In [ ]:
# mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)

# plt.figure(figsize=(10,4))
# librosa.display.specshow(mfcc, x_axis='time')
# plt.colorbar()
# plt.title("MFCC Example")
# plt.show()


In [ ]:
# silence_count=0

# for genre in GENRES:

#     songs=os.listdir(f"{GENRES_PATH}/{genre}")[:20]

#     for song in songs:

#         file=f"{GENRES_PATH}/{genre}/{song}/vocals.wav"

#         y,sr=librosa.load(file,sr=None)

#         if np.max(np.abs(y[:int(0.5*sr)]))<1e-4:

#             silence_count+=1

# print("Silence stems:",silence_count)

In [ ]:
# def silence_ratio(file_path):
#     y, sr = librosa.load(file_path, sr=None)
#     intervals = librosa.effects.split(y, top_db=20)
#     voiced = sum((end-start) for start,end in intervals)
#     return 1 - voiced/len(y)

# ratios = []
# for genre in os.listdir(GENRES_PATH):
#     songs = os.listdir(f"{GENRES_PATH}/{genre}")[:20]
#     for song in songs:
#         path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
#         if os.path.exists(path):
#             ratios.append(silence_ratio(path))

# plt.hist(ratios, bins=20)
# plt.title("Silence Ratio Distribution")
# plt.show()


In [ ]:
# corrupted=0
# small=0

# for genre in GENRES:
#     for song in os.listdir(f"{GENRES_PATH}/{genre}"):
#         for stem in STEMS:
#             fp=f"{GENRES_PATH}/{genre}/{song}/{stem}"

#             if os.path.exists(fp):

#                 size=os.path.getsize(fp)

#                 if size<4096:
#                     corrupted+=1

#                 if size<5*1024*1024:
#                     small+=1

# print("Corrupted files:",corrupted)
# print("Small files:",small)

In [ ]:
# noise_lengths = []
# noise_sample_rates = []

# for nf in tqdm(noise_files[:200]):   
#     y, sr = librosa.load(nf, sr=None)
#     noise_lengths.append(len(y)/sr)
#     noise_sample_rates.append(sr)

# plt.figure(figsize=(8,4))
# plt.hist(noise_lengths, bins=20)
# plt.title("ESC-50 Noise Duration Distribution")
# plt.xlabel("Seconds")
# plt.ylabel("Count")
# plt.show()

# print("Mean noise length:", np.mean(noise_lengths))
# print("Sample rate distribution:")
# print(pd.Series(noise_sample_rates).value_counts())

In [ ]:
# noise_example = np.random.choice(noise_files)

# y, sr = librosa.load(noise_example, sr=None)

# plt.figure(figsize=(12,4))
# librosa.display.waveshow(y, sr=sr)
# plt.title("Example ESC-50 Noise")
# plt.show()

# spec = librosa.feature.melspectrogram(y=y, sr=sr)
# spec_db = librosa.power_to_db(spec, ref=np.max)

# plt.figure(figsize=(12,4))
# librosa.display.specshow(spec_db, sr=sr, x_axis="time", y_axis="mel")
# plt.title("ESC-50 Noise Spectrogram")
# plt.colorbar()
# plt.show()

In [ ]:
# missing = []

# for genre in GENRES:
#     songs = os.listdir(f"{GENRES_PATH}/{genre}")
    
#     for song in songs:
#         path = f"{GENRES_PATH}/{genre}/{song}"
        
#         for stem in STEMS:
#             fp = f"{path}/{stem}"
            
#             if not os.path.exists(fp):
#                 missing.append(fp)

# print("Missing stems:", len(missing))

In [ ]:
# energy = []

# for genre in GENRES:
    
#     songs = os.listdir(f"{GENRES_PATH}/{genre}")[:30]
    
#     for song in songs:
        
#         file = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
        
#         y, sr = librosa.load(file, sr=None)
        
#         rms = np.mean(librosa.feature.rms(y=y))
        
#         energy.append(rms)

# plt.hist(energy, bins=20)
# plt.title("Audio Energy Distribution")
# plt.xlabel("RMS Energy")
# plt.show()

### Dataset Observations

• The dataset contains 10 music genres: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, and rock.

• Each genre contains approximately 100 songs, indicating that the dataset is balanced across classes.

• The average audio length is around 30 seconds, with very little variation.

• All audio files share a sampling rate of 44.1 kHz.

• Each song contains four stems: drums, vocals, bass, and other. No missing stems were detected.

• The ESC-50 dataset provides 2000 environmental noise clips with an average duration of 5 seconds.

• Silence analysis shows that some audio segments contain large silent regions, which may need filtering during preprocessing.

• MFCC visualizations indicate different spectral patterns between genres, suggesting that spectral features can help distinguish music styles.

## Feature Extraction

In [ ]:
# SR = 22050

# def extract_features(segment):

#     segment = segment / (np.max(np.abs(segment)) + 1e-6)

#     mfcc = librosa.feature.mfcc(y=segment, sr=SR, n_mfcc=40)
#     mfcc_mean = np.mean(mfcc, axis=1)
#     mfcc_std = np.std(mfcc, axis=1)

#     delta = librosa.feature.delta(mfcc)
#     delta2 = librosa.feature.delta(mfcc, order=2)

#     delta_mean = np.mean(delta, axis=1)
#     delta2_mean = np.mean(delta2, axis=1)

#     chroma = librosa.feature.chroma_stft(y=segment, sr=SR)
#     chroma_mean = np.mean(chroma, axis=1)

#     contrast = librosa.feature.spectral_contrast(y=segment, sr=SR)
#     contrast_mean = np.mean(contrast, axis=1)

#     flatness = float(np.mean(librosa.feature.spectral_flatness(y=segment)))

#     centroid = float(np.mean(librosa.feature.spectral_centroid(y=segment, sr=SR)))
#     rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=segment, sr=SR)))
#     bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(y=segment, sr=SR)))

#     tempo,_ = librosa.beat.beat_track(y=segment, sr=SR)
#     tempo = float(np.squeeze(tempo))

#     rms = float(np.mean(librosa.feature.rms(y=segment)))
#     zcr = float(np.mean(librosa.feature.zero_crossing_rate(segment)))

#     flux = float(np.mean(librosa.onset.onset_strength(y=segment, sr=SR)))

#     return np.concatenate([
#         mfcc_mean,
#         mfcc_std,
#         delta_mean,
#         delta2_mean,
#         chroma_mean,
#         contrast_mean,
#         np.array([
#             centroid,
#             rolloff,
#             bandwidth,
#             tempo,
#             flatness,
#             rms,
#             zcr,
#             flux
#         ])
#     ])

In [ ]:
# noise_cache = []

# for nf in tqdm(noise_files):
#     y,_ = librosa.load(nf, sr=SR)
#     noise_cache.append(y)

# print("Loaded noise samples:",len(noise_cache))

## Build Training Dataset

In [ ]:
# def random_stem_mix(genre):

#     songs = os.listdir(f"{GENRES_PATH}/{genre}")

#     stems_audio = []

#     for stem in STEMS:

#         song = random.choice(songs)

#         file = f"{GENRES_PATH}/{genre}/{song}/{stem}"

#         y_audio,_ = librosa.load(file, sr=SR)

#         stems_audio.append(y_audio)

#     min_len = min(len(s) for s in stems_audio)

#     stems_audio = [s[:min_len] for s in stems_audio]

#     mix = np.mean(stems_audio, axis=0)

#     return mix

In [ ]:
# X=[]
# y=[]
# song_ids=[]

# SEGMENT_DURATION=5
# HOP_DURATION=2.5

# segment_len=int(SEGMENT_DURATION*SR)
# hop_len=int(HOP_DURATION*SR)

# for genre in GENRES:

#     songs=os.listdir(f"{GENRES_PATH}/{genre}")

#     for song in tqdm(songs):

#         song_path=f"{GENRES_PATH}/{genre}/{song}"

#         song_id=f"{genre}_{song}"

#         if np.random.rand() < 0.5:

#             signals=[
#                 librosa.load(f"{song_path}/{stem}", sr=SR)[0]
#                 for stem in STEMS
#             ]

#             mix=np.mean(signals,axis=0)

#         else:

#             mix=random_stem_mix(genre)


#         start=0

#         while start+segment_len<=len(mix):

#             segment=mix[start:start+segment_len]

#             start+=hop_len

#             if np.max(np.abs(segment))<1e-4:
#                 continue

#             segment=segment/(np.max(np.abs(segment))+1e-6)

#             try:

#                 feat=extract_features(segment)

#                 X.append(feat)
#                 y.append(genre)
#                 song_ids.append(song_id)

#             except:
#                 continue


            
#             for _ in range(3):

#                 noise=random.choice(noise_cache)

#                 max_start=len(noise)-segment_len

#                 if max_start<=0:
#                     continue

#                 idx=np.random.randint(0,max_start)

#                 noise_segment=noise[idx:idx+segment_len]

#                 noise_segment=noise_segment/(np.max(np.abs(noise_segment))+1e-6)

#                 noise_strength=np.random.uniform(0.05,0.2)

#                 noisy_segment=segment+noise_strength*noise_segment

#                 noisy_segment=noisy_segment/(np.max(np.abs(noisy_segment))+1e-6)

#                 try:

#                     feat_noise=extract_features(noisy_segment)

#                     X.append(feat_noise)
#                     y.append(genre)
#                     song_ids.append(song_id)

#                 except:
#                     continue


# X=np.array(X)
# y=np.array(y)
# song_ids=np.array(song_ids)

# print("Dataset size:",X.shape)

## Classical ML

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()

# y_encoded = le.fit_transform(y)

# print("Classes:", le.classes_)

In [ ]:
# from sklearn.model_selection import GroupShuffleSplit

# gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

# train_idx, val_idx = next(gss.split(X, y, groups=song_ids))

# X_train = X[train_idx]
# X_val = X[val_idx]

# y_train = y_encoded[train_idx]
# y_val = y_encoded[val_idx]

In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)

In [ ]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score

# lr_model = LogisticRegression(max_iter=2000)

# lr_model.fit(X_train, y_train)

# lr_preds = lr_model.predict(X_val)

# lr_f1 = f1_score(y_val, lr_preds, average="macro")

# print("Logistic Regression Macro F1:", lr_f1)

In [ ]:
# from sklearn.naive_bayes import GaussianNB

# nb_model = GaussianNB()

# nb_model.fit(X_train, y_train)

# nb_preds = nb_model.predict(X_val)

# nb_f1 = f1_score(y_val, nb_preds, average="macro")

# print("Naive Bayes Macro F1:", nb_f1)

In [ ]:
# from lightgbm import LGBMClassifier

# lgb_model = LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=8,
#     num_leaves=64
# )

# lgb_model.fit(X_train, y_train)

# lgb_preds = lgb_model.predict(X_val)

# lgb_f1 = f1_score(y_val, lgb_preds, average="macro")

# print("LightGBM Macro F1:", lgb_f1)

In [ ]:
# from xgboost import XGBClassifier

# xgb = XGBClassifier(
#     n_estimators=500,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="multi:softmax",
#     num_class=10,
#     tree_method="hist",
#     random_state=42
# )

# xgb.fit(X_train, y_train)

# xgb_preds = xgb.predict(X_val)

# xgb_f1 = f1_score(y_val, xgb_preds, average="macro")

# print("XGBoost Macro F1:", xgb_f1)


In [ ]:
# from catboost import CatBoostClassifier

# cat = CatBoostClassifier(
#     iterations=500,
#     depth=6,
#     learning_rate=0.05,
#     loss_function="MultiClass",
#     verbose=False
# )

# cat.fit(X_train, y_train)

# cat_preds = cat.predict(X_val)

# cat_f1 = f1_score(y_val, cat_preds, average="macro")

# print("CatBoost Macro F1:", cat_f1)


In [ ]:
# results = pd.DataFrame({
#     "Model": [
#         "Logistic Regression",
#         "Naive Bayes",
#         "LightGBM",
#         "XGBoost",
#         "CatBoost"
#     ],
#     "Macro F1": [
#         lr_f1,
#         nb_f1,
#         lgb_f1,
#         xgb_f1,
#         cat_f1
#     ]
# })

# results = results.sort_values("Macro F1", ascending=False)

# results

In [ ]:
# plt.figure(figsize=(8,4))

# sns.barplot(
#     data=results,
#     x="Macro F1",
#     y="Model"
# )

# plt.title("Classical ML Model Comparison (Macro F1)")
# plt.show()

In [ ]:
# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# cm = confusion_matrix(y_val, lgb_preds)

# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
#             xticklabels=le.classes_,
#             yticklabels=le.classes_)
# plt.title("Confusion Matrix - LightGBM")
# plt.show()

In [ ]:
# X_test = []

# for fname in tqdm(test_df["filename"]):

#     file_path = f"{DATA_PATH}/{fname}"

#     y_audio,_ = librosa.load(file_path, sr=SR)

#     segment_len = SEGMENT_DURATION * SR

#     segments = len(y_audio) // segment_len

#     feats = []

#     for i in range(segments):

#         segment = y_audio[
#             i*segment_len:(i+1)*segment_len
#         ]

#         feat = extract_features(segment)

#         feats.append(feat)

#     X_test.append(np.mean(feats, axis=0))

# X_test = scaler.transform(np.array(X_test))

In [ ]:
# import wandb

# wandb.init(
#     project="24f1001527-t12026",
#     name="classical_ml_baseline"
# )
# wandb.log({
#     "LogisticRegression_F1": lr_f1,
#     "NaiveBayes_F1": nb_f1,
#     "LightGBM_F1": lgb_f1,
#     "XGBoost_F1": xgb_f1,
#     "CatBoost_F1": cat_f1
# })

# wandb.log({"Model Comparison": wandb.Table(dataframe=results)})

# wandb.finish()

##  Classical ML Model Comparison

| Model                | Macro F1 Score |
|---------------------|---------------|
| Logistic Regression | 0.81          |
| Naive Bayes         | 0.70          |
| SVM (RBF)           | 0.79          |
| Random Forest       | 0.76          |
| XGBoost / LightGBM  | 0.82          |

In [ ]:
# preds = lgb_model.predict(X_test)

# pred_labels = le.inverse_transform(preds)

In [ ]:
# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": pred_labels
# })

# submission.to_csv("submission.csv", index=False)

# submission.head()

## Neural Network

In [ ]:
SR = 22050
N_MELS = 128

def audio_to_mel(audio):

    mel1 = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_mels=N_MELS,
        hop_length=512,
        n_fft=2048
    )

    mel2 = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_mels=N_MELS,
        hop_length=256,
        n_fft=1024
    )

    mel1 = librosa.power_to_db(mel1, ref=np.max)
    mel2 = librosa.power_to_db(mel2, ref=np.max)

    mel1 = (mel1 - mel1.mean()) / (mel1.std() + 1e-6)
    mel2 = (mel2 - mel2.mean()) / (mel2.std() + 1e-6)

    
    mel2 = librosa.util.fix_length(mel2, size=mel1.shape[1], axis=1)

    mel = np.stack([mel1, mel2], axis=0)

    return mel

In [ ]:
os.makedirs("mel_dataset", exist_ok=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(GENRES)

In [ ]:
NUM_MIXES = 3
SR = 22050
SEGMENT_DURATION = 5
HOP_DURATION = 1.0

segment_len = int(SEGMENT_DURATION * SR)
hop_len = int(HOP_DURATION * SR)


In [ ]:
meta = []

for genre in GENRES:
    genre_songs = sorted(os.listdir(os.path.join(GENRES_PATH, genre)))

    for song in genre_songs:
        song_id = f"{genre}_{song}"

        for mix_id in range(NUM_MIXES):
            meta.append({
                "genre": genre,
                "song_id": song_id,
                "songs": genre_songs
            })

In [ ]:
class GenreDataset(Dataset):

    def __init__(self, meta, GENRES_PATH, ESC_PATH, augment=False):
        self.meta = meta
        self.GENRES_PATH = GENRES_PATH
        self.augment = augment

        self.STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

        self.loaded_noise = []
        for f in os.listdir(ESC_PATH):
            y, _ = librosa.load(os.path.join(ESC_PATH, f), sr=SR)
            self.loaded_noise.append(y)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):

        row = self.meta[idx]
        genre = row["genre"]
        genre_songs = row["songs"]

        signals = []

        for stem in self.STEMS:
            rand_song = random.choice(genre_songs)
            stem_path = os.path.join(self.GENRES_PATH, genre, rand_song, stem)

            y, _ = librosa.load(stem_path, sr=SR)

            if np.random.rand() < 0.5:
                rate = np.random.uniform(0.9, 1.1)
                y = librosa.effects.time_stretch(y, rate)

            signals.append(y)

        if np.random.rand() < 0.3:
            drop_idx = np.random.randint(0, len(signals))
            signals[drop_idx] = np.zeros_like(signals[drop_idx])

        min_len = min(len(s) for s in signals)
        signals = [s[:min_len] for s in signals]

        mix = np.mean(signals, axis=0)

        if len(mix) > segment_len:
            start = np.random.randint(0, len(mix) - segment_len)
            segment = mix[start:start + segment_len]
        else:
            segment = np.pad(mix, (0, segment_len - len(mix)))

        segment = segment / (np.max(np.abs(segment)) + 1e-6)

        gain = np.random.uniform(0.8, 1.2)
        segment *= gain

        if np.random.rand() < 0.4:
            n = random.choice(self.loaded_noise)

            if len(n) >= len(segment):
                idx = np.random.randint(0, len(n) - len(segment) + 1)
                n = n[idx:idx + len(segment)]
            else:
                n = np.pad(n, (0, len(segment) - len(n)))[:len(segment)]

            strength = np.random.uniform(0.03, 0.25)
            segment += strength * n

        if np.random.rand() < 0.3:
            steps = np.random.uniform(-1, 1)
            segment = librosa.effects.pitch_shift(segment, sr=SR, n_steps=steps)

        segment = segment / (np.max(np.abs(segment)) + 1e-6)

        mel = audio_to_mel(segment)

        if self.augment:
            t = np.random.randint(20, 50)
            t0 = np.random.randint(0, mel.shape[2] - t)
            mel[:, :, t0:t0+t] = 0

            f = np.random.randint(5, 20)
            f0 = np.random.randint(0, mel.shape[1] - f)
            mel[:, f0:f0+f, :] = 0

            mel = mel + np.random.normal(0, 0.01, mel.shape)

        mel = torch.tensor(mel, dtype=torch.float32)
        label = torch.tensor(GENRES.index(genre))

        return mel, label

In [ ]:
train_dataset = GenreDataset(meta, GENRES_PATH, ESC_PATH, augment=True)
val_dataset = GenreDataset(meta, GENRES_PATH, ESC_PATH, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

## CNN

In [ ]:
class AudioCNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(2, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,num_classes)
        )

    def forward(self,x):

        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)

        return x

In [ ]:
# model = AudioCNN().to(device)

# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=3e-4,
#     weight_decay=1e-4
# )

In [ ]:
# import wandb

# wandb.init(
#     project="24f1001527-t12026",
#     name="Simple_CNN"
# )

In [ ]:
# artifact_name = "cnn_genre_model"

# try:
#     artifact = wandb.use_artifact(
#         "24f1001527-dl-genai-project/24f1001527-t12026/cnn_genre_model:latest",
#         type="model"
#     )

#     artifact_dir = artifact.download()

#     model_path = os.path.join(artifact_dir, "best_cnn_model.pth")

#     checkpoint = torch.load(model_path, map_location=device)

#     model.load_state_dict(checkpoint["model_state"])
#     optimizer.load_state_dict(checkpoint["optimizer_state"])

#     best_f1 = checkpoint["best_f1"]
#     prev_epochs = checkpoint["epochs_trained"]

#     print("Loaded previous CNN checkpoint")
#     print("Previous epochs:", prev_epochs)

# except Exception as e:

#     best_f1 = 0.0
#     prev_epochs = 0

#     print("Training CNN from scratch")
#     print("Reason:", e)

In [ ]:
# NEW_EPOCHS = 5
# total_epochs = prev_epochs + NEW_EPOCHS

# alpha = 0.2

# 
# for g in optimizer.param_groups:
#     g["lr"] = 1e-4

# 
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max=NEW_EPOCHS
# )

# for epoch in range(NEW_EPOCHS):

#     current_epoch = prev_epochs + epoch + 1

#     model.train()
#     train_loss = 0

#     for mel, label in train_loader:

#         mel = mel.to(device)
#         label = label.to(device).long()

#         optimizer.zero_grad()

#     
#         if torch.rand(1).item() < 0.5:

#             lam = np.random.beta(alpha, alpha)

#             index = torch.randperm(mel.size(0)).to(device)

#             mixed_mel = lam * mel + (1 - lam) * mel[index]

#             label_a = label
#             label_b = label[index]

#             output = model(mixed_mel)

#             loss = lam * criterion(output, label_a) + (1 - lam) * criterion(output, label_b)

#         else:

#             output = model(mel)

#             loss = criterion(output, label)

#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         optimizer.step()

#         train_loss += loss.item()

#     train_loss /= len(train_loader)

#     
#     scheduler.step()

#     model.eval()

#     preds = []
#     targets = []

#     with torch.no_grad():

#         for mel, label in val_loader:

#             mel = mel.to(device)
#             label = label.to(device).long()

#             output = model(mel)

#             pred = torch.argmax(output, dim=1)

#             preds.extend(pred.cpu().numpy())
#             targets.extend(label.cpu().numpy())

#     val_f1 = f1_score(targets, preds, average="macro")

#     if val_f1 > best_f1:

#         best_f1 = val_f1

#         torch.save({
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_f1": best_f1,
#             "epochs_trained": current_epoch
#         }, "best_cnn_model.pth")

#     wandb.log({
#         "epoch": current_epoch,
#         "train_loss": train_loss,
#         "val_macro_f1": val_f1
#     })

#     print(f"Epoch {current_epoch} | Train Loss {train_loss:.4f} | F1 {val_f1:.4f}")

In [ ]:
# if not os.path.exists("best_cnn_model.pth"):
#     torch.save({
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "best_f1": best_f1,
#         "epochs_trained": total_epochs
#     }, "best_cnn_model.pth")


# artifact = wandb.Artifact(
#     name="cnn_genre_model",
#     type="model",
#     metadata={
#         "val_macro_f1": float(best_f1),
#         "epochs_trained": int(total_epochs),
#         "model": "CNN"
#     }
# )

# artifact.add_file("best_cnn_model.pth")

# logged_artifact = wandb.log_artifact(artifact)
# logged_artifact.wait()

# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/cnn_genre_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# model_path = artifact_dir + "/best_cnn_model.pth"

# checkpoint = torch.load("best_cnn_model.pth", map_location=device, weights_only=False)

# model.load_state_dict(checkpoint["model_state"])
# model.eval()

# cnn_f1 = checkpoint.get("best_f1")
# epochs_trained = checkpoint.get("epochs_trained")

# print("Best CNN Validation F1:", cnn_f1)
# print("Total Epochs Trained:", epochs_trained)

# wandb.finish()

## CRNN

In [ ]:
class CRNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(2, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.rnn = nn.GRU(
            input_size=2048,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,num_classes)
        )


    def forward(self,x):

        x = self.cnn(x)

        B,C,H,W = x.shape

        x = x.permute(0,3,1,2)

        x = x.reshape(B,W,C*H)

        out,_ = self.rnn(x)

        x = torch.mean(out, dim=1)

        x = self.classifier(x)

        return x

In [ ]:
# import wandb

# wandb.init(
#     project="24f1001527-t12026",
#     name="CRNN_GRU"
# )

In [ ]:
# model = CRNN().to(device)

# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=3e-4,
#     weight_decay=1e-4
# )

In [ ]:
# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/crnn_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# model_path = artifact_dir + "/best_crnn_model.pth"

# checkpoint = torch.load(model_path, map_location=device,weights_only=False)

# model.load_state_dict(checkpoint["model_state"])
# model.eval()

# crnn_f1 = checkpoint.get("best_f1")
# epochs_trained = checkpoint.get("epochs_trained")

# print("Best CRNN Validation F1:", crnn_f1)
# print("Total Epochs Trained:", epochs_trained)

In [ ]:
# optimizer.load_state_dict(checkpoint["optimizer_state"])

In [ ]:
# for g in optimizer.param_groups:
#     g["lr"] = 1e-5

In [ ]:
# criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)

In [ ]:
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=64,
#     shuffle=True,
#     num_workers=2
# )

In [ ]:
# val_loader = DataLoader(
#     val_dataset,
#     batch_size=64,
#     shuffle=False,
#     num_workers=2
# )

In [ ]:
# NEW_EPOCHS = 5
# alpha = 0.1

# for epoch in range(NEW_EPOCHS):

#     current_epoch = epoch + 1

#     model.train()
#     train_loss = 0

#     for mel, label in train_loader:

#         mel = mel.to(device)
#         label = label.to(device).long()

#         optimizer.zero_grad()

#         if torch.rand(1).item() < 0.5:

#             lam = np.random.beta(alpha, alpha)

#             index = torch.randperm(mel.size(0)).to(device)

#             mixed_mel = lam * mel + (1 - lam) * mel[index]

#             label_a = label
#             label_b = label[index]

#             output = model(mixed_mel)

#             loss = lam * criterion(output, label_a) + (1 - lam) * criterion(output, label_b)

#         else:

#             output = model(mel)

#             loss = criterion(output, label)

#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         optimizer.step()

#         train_loss += loss.item()

#     train_loss /= len(train_loader)

#     model.eval()

#     preds = []
#     targets = []

#     with torch.no_grad():

#         for mel, label in val_loader:

#             mel = mel.to(device)
#             label = label.to(device).long()

#             output = model(mel)

#             pred = torch.argmax(output, dim=1)

#             preds.extend(pred.cpu().numpy())
#             targets.extend(label.cpu().numpy())

#     val_f1 = f1_score(targets, preds, average="macro")

#     if val_f1 > best_f1:

#         best_f1 = val_f1

#         torch.save({
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_f1": best_f1,
#             "epochs_trained": current_epoch
#         }, "best_crnn_model.pth")

#     wandb.log({
#         "epoch": current_epoch,
#         "train_loss": train_loss,
#         "val_macro_f1": val_f1
#     })

#     print(f"Epoch {current_epoch} | F1 {val_f1:.4f}")

In [ ]:
# artifact = wandb.Artifact(
#     name="crnn_model",
#     type="model",
#     metadata={
#         "val_macro_f1": float(best_f1),
#         "epochs_trained": NEW_EPOCHS,
#         "model": "CRNN"
#     }
# )

# artifact.add_file("best_crnn_model.pth")

# wandb.log_artifact(artifact)

In [ ]:
# checkpoint = torch.load("best_crnn_model.pth", map_location=device,weights_only=False)

# model.load_state_dict(checkpoint["model_state"])
# model.eval()

# crnn_f1 = checkpoint["best_f1"]

# print("Best CRNN Validation F1:", crnn_f1)

In [ ]:
# comparison = pd.DataFrame({
#     "Model":["CNN","CRNN"],
#     "Macro F1":[cnn_f1, crnn_f1]
# })

# print(comparison)

# sns.barplot(data=comparison, x="Model", y="Macro F1")

## Transformers

In [ ]:
# wandb.init(
#     project="24f1001527-t12026",
#     name="HuBERT_Finetune"
# )

In [ ]:
# feature_extractor = AutoFeatureExtractor.from_pretrained(
#     "facebook/hubert-base-ls960"
# )

# model = HubertForSequenceClassification.from_pretrained(
#     "facebook/hubert-base-ls960",
#     num_labels=len(le.classes_)
# )

# model.to(device)

In [ ]:
# SR_CNN = 22050
# SR_TRANSFORMER = 16000

In [ ]:
# class TransformerDataset(Dataset):

#     def __init__(self, audio_data, labels):
#         self.audio = audio_data
#         self.labels = labels

#     def __len__(self):
#         return len(self.audio)

#     def __getitem__(self, idx):

#         audio = self.audio[idx]

#         inputs = feature_extractor(
#             audio,
#             sampling_rate=feature_extractor.sampling_rate,
#             return_tensors="pt",
#             padding="max_length",
#             truncation=True,
#             max_length=feature_extractor.sampling_rate * 5
#         )

#         item = {
#             "input_values": inputs.input_values.squeeze(0),
#             "labels": torch.tensor(self.labels[idx])
#         }

#         return item

In [ ]:
# train_dataset = TransformerDataset(X_audio[train_idx], y_train)
# val_dataset = TransformerDataset(X_audio[val_idx], y_val)

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=2e-5
# )

# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
# artifact_name = "hubert_model"

# try:

#     artifact = wandb.use_artifact(
#         f"24f1001527-dl-genai-project/24f1001527-t12026/{artifact_name}:latest",
#         type="model"
#     )

#     artifact_dir = artifact.download()

#     checkpoint = torch.load(
#         artifact_dir + "/best_hubert_model.pth",
#         map_location=device
#     )

#     model.load_state_dict(checkpoint["model_state"])
#     optimizer.load_state_dict(checkpoint["optimizer_state"])

#     best_f1 = checkpoint["best_f1"]
#     prev_epochs = checkpoint["epochs_trained"]

#     print("Loaded previous HuBERT checkpoint")

# except:

#     best_f1 = 0
#     prev_epochs = 0
#     print("Training HuBERT from scratch")

In [ ]:
# NEW_EPOCHS = 0
# total_epochs = prev_epochs + NEW_EPOCHS

# for epoch in range(NEW_EPOCHS):

#     current_epoch = prev_epochs + epoch + 1

#     model.train()
#     train_loss = 0

#     for batch in train_loader:

#         inputs = batch["input_values"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()

#         outputs = model(input_values=inputs)

#         logits = outputs.logits

#         loss = criterion(logits, labels)

#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()

#     train_loss /= len(train_loader)
    
#     model.eval()
    
#     preds = []
#     targets = []
    
#     with torch.no_grad():
    
#         for batch in val_loader:
    
#             inputs = batch["input_values"].to(device)
#             labels = batch["labels"].to(device)
    
#             outputs = model(input_values=inputs)
    
#             pred = torch.argmax(outputs.logits, dim=1)
    
#             preds.extend(pred.cpu().numpy())
#             targets.extend(labels.cpu().numpy())
    
#     val_f1 = f1_score(targets, preds, average="macro")
    
#     if val_f1 > best_f1:
    
#         best_f1 = val_f1
    
#         torch.save({
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_f1": best_f1,
#             "epochs_trained": current_epoch
#         }, "best_hubert_model.pth")
#     print(f"Epoch {current_epoch} | F1 {val_f1:.4f}")

In [ ]:
# if not os.path.exists("best_hubert_model.pth"):
#     torch.save({
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "best_f1": best_f1,
#         "epochs_trained": current_epoch
#     }, "best_hubert_model.pth")

In [ ]:
# artifact = wandb.Artifact(
#     name="hubert_model",
#     type="model",
#     metadata={
#         "val_macro_f1": float(best_f1),
#         "epochs_trained": int(total_epochs),
#         "model": "HuBERT"
#     }
# )

# artifact.add_file("best_hubert_model.pth")

# wandb.log_artifact(artifact)

In [ ]:
# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/hubert_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# model_path = artifact_dir + "/best_hubert_model.pth"

# checkpoint = torch.load(model_path, map_location=device)

# model.load_state_dict(checkpoint["model_state"])

# model.eval()

# hubert_f1 = checkpoint.get("best_f1")

# epochs_trained = checkpoint.get("epochs_trained")

# print("Best HuBERT Validation F1:", hubert_f1)
# print("Total Epochs Trained:", epochs_trained)

In [ ]:
# comparison = pd.DataFrame({
#     "Model":["CNN","CRNN","HuBERT"],
#     "Macro F1":[cnn_f1, crnn_f1, hubert_f1]
# })

In [ ]:
# sns.barplot(data=comparison,x="Model",y="Macro F1")

In [ ]:
# wandb.finish()

## Resnet

In [ ]:
# import wandb

# wandb.init(
#     project="24f1001527-t12026",
#     name="ResNet18_MelSpectrogram"
# )

In [ ]:
from torchvision import models

In [ ]:
class AudioResNet(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.resnet = models.resnet18(weights="IMAGENET1K_V1")

        self.resnet.conv1 = nn.Conv2d(
            2,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.resnet.fc = nn.Linear(
            self.resnet.fc.in_features,
            num_classes
        )

    def forward(self, x):
        return self.resnet(x)

In [ ]:
# model = AudioResNet().to(device)

# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=3e-4,
#     weight_decay=1e-4
# )

In [ ]:
# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/resnet_genre_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# model_path = artifact_dir + "/best_resnet_model.pth"

# checkpoint = torch.load(model_path, map_location=device,weights_only=False)

# model.load_state_dict(checkpoint["model_state"])

In [ ]:
# for g in optimizer.param_groups:
#     g["lr"] = 1e-5

In [ ]:
# NEW_EPOCHS = 6
# prev_epoch=20
# total_epochs = prev_epochs + NEW_EPOCHS

# alpha = 0.2

# for epoch in range(NEW_EPOCHS):

#     current_epoch = prev_epochs + epoch + 1

#     model.train()
#     train_loss = 0

#     for mel, label in train_loader:

#         mel = mel.to(device)
#         label = label.to(device).long()

#         optimizer.zero_grad()

#         if torch.rand(1).item() < 0.5:

#             lam = np.random.beta(alpha, alpha)

#             index = torch.randperm(mel.size(0)).to(device)

#             mixed_mel = lam * mel + (1 - lam) * mel[index]

#             label_a = label
#             label_b = label[index]

#             output = model(mixed_mel)

#             loss = lam * criterion(output, label_a) + (1 - lam) * criterion(output, label_b)

#         else:

#             output = model(mel)

#             loss = criterion(output, label)

#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         optimizer.step()

#         train_loss += loss.item()

#     train_loss /= len(train_loader)

#     model.eval()

#     preds = []
#     targets = []

#     with torch.no_grad():

#         for mel, label in val_loader:

#             mel = mel.to(device)
#             label = label.to(device).long()

#             output = model(mel)

#             pred = torch.argmax(output, dim=1)

#             preds.extend(pred.cpu().numpy())
#             targets.extend(label.cpu().numpy())

#     val_f1 = f1_score(targets, preds, average="macro")

#     if val_f1 > best_f1:

#         best_f1 = val_f1

#         torch.save({
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "best_f1": best_f1,
#             "epochs_trained": current_epoch
#         }, "best_resnet_model.pth")

#     wandb.log({
#         "epoch": current_epoch,
#         "train_loss": train_loss,
#         "val_macro_f1": val_f1
#     })

#     print(f"Epoch {current_epoch} | Train Loss {train_loss:.4f} | F1 {val_f1:.4f}")

In [ ]:
# import os

# if not os.path.exists("best_resnet_model.pth"):
#     torch.save({
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "best_f1": best_f1,
#         "epochs_trained": current_epoch
#     }, "best_resnet_model.pth")

In [ ]:
# artifact = wandb.Artifact(
#     name="resnet_genre_model",
#     type="model",
#     metadata={
#         "val_macro_f1": float(best_f1),
#         "epochs_trained": int(total_epochs),
#         "model": "ResNet18"
#     }
# )

# artifact.add_file("best_resnet_model.pth")

# logged_artifact = wandb.log_artifact(artifact)
# logged_artifact.wait()

In [ ]:
# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/resnet_genre_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# model_path = artifact_dir + "/best_resnet_model.pth"

# checkpoint = torch.load(model_path, map_location=device,weights_only=False)

# model.load_state_dict(checkpoint["model_state"])

# model.eval()

# resnet_f1 = checkpoint.get("best_f1")
# epochs_trained = checkpoint.get("epochs_trained")

# print("Best ResNet Validation F1:", resnet_f1)
# print("Total Epochs Trained:", epochs_trained)

In [ ]:
# wandb.finish()

In [ ]:
# comparison = pd.DataFrame({
#     "Model":["CNN","CRNN","ResNet"],
#     "Macro F1":[cnn_f1, crnn_f1, resnet_f1]
# })

# comparison

In [ ]:
# sns.barplot(data=comparison, x="Model", y="Macro F1")

In [ ]:
# wandb.init(
#     project="24f1001527-t12026",
#     name="submission_run"
# )
# cnn_model = AudioCNN().to(device)

# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/cnn_genre_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# checkpoint = torch.load(
#     f"{artifact_dir}/best_cnn_model.pth",
#     map_location=device,
#     weights_only=False
# )

# cnn_model.load_state_dict(checkpoint["model_state"])
# cnn_model.eval()

# print("CNN Best F1:", checkpoint["best_f1"])


# crnn_model = CRNN().to(device)

# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/crnn_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# checkpoint = torch.load(
#     f"{artifact_dir}/best_crnn_model.pth",
#     map_location=device,
#     weights_only=False
# )

# crnn_model.load_state_dict(checkpoint["model_state"])
# crnn_model.eval()

# print("CRNN Best F1:", checkpoint["best_f1"])



# resnet_model = AudioResNet().to(device)

# artifact = wandb.use_artifact(
#     "24f1001527-dl-genai-project/24f1001527-t12026/resnet_genre_model:latest",
#     type="model"
# )

# artifact_dir = artifact.download()

# checkpoint = torch.load(
#     f"{artifact_dir}/best_resnet_model.pth",
#     map_location=device,
#     weights_only=False
# )

# resnet_model.load_state_dict(checkpoint["model_state"])
# resnet_model.eval()

# print("ResNet Best F1:", checkpoint["best_f1"])


# print("\nAll models loaded successfully.")


In [ ]:
# temperature = 0.8

# predictions = []

# cnn_model.eval()
# crnn_model.eval()
# resnet_model.eval()

# with torch.no_grad():

#     for fname in tqdm(test_df["filename"]):

#         path = f"{DATA_PATH}/{fname}"

#         audio, _ = librosa.load(path, sr=SR)

#         if len(audio) < segment_len:
#             audio = np.pad(audio, (0, segment_len - len(audio)))
#         segments = []

#         start = 0

#         while start + segment_len <= len(audio):

#             segment = audio[start:start + segment_len]
#             start += hop_len

#             segment = segment / (np.max(np.abs(segment)) + 1e-6)

#             mel = audio_to_mel(segment)

#             mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).to(device)

#             cnn_pred = torch.softmax(
#                 cnn_model(mel_tensor) / temperature,
#                 dim=1
#             )

#             crnn_pred = torch.softmax(
#                 crnn_model(mel_tensor),
#                 dim=1
#             )

#             resnet_pred = torch.softmax(
#                 resnet_model(mel_tensor) / temperature,
#                 dim=1
#             )

#             pred = (
#                 0.6 * resnet_pred +
#                 0.25 * crnn_pred +
#                 0.15 * cnn_pred
#             )

#             segments.append(pred.cpu().numpy()[0])

#         if len(segments) == 0:
#             segments = np.zeros(len(le.classes_))
#         else:
#             segments = np.array(segments)
#             segments = np.mean(segments, axis=0)

#         predictions.append(np.argmax(segments))


# pred_labels = le.inverse_transform(predictions)

# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": pred_labels
# })

# submission.to_csv("submission.csv", index=False)

# submission.head()

In [ ]:
wandb.finish()

| Model  | Macro F1 Score |
|--------|---------------|
| CNN    | 0.9303        |
| CRNN   | 0.9360        |
| ResNet | 0.9534        |

## Milestone 1


In [ ]:
# import os
# import glob
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# import librosa
# import librosa.display
# import matplotlib.pyplot as plt
# import random
# import torch
# import warnings
# warnings.filterwarnings("ignore")


# DATA_SEED = 67
# TRAINING_SEED = 1234
# SR = 22050
# DURATION = 5.0
# N_FFT = 2048
# HOP_LENGTH = 512
# N_MELS = 128
# TOP_DB=20
# TARGET_SNR_DB = 10

# random.seed(DATA_SEED)
# np.random.seed(DATA_SEED)
# torch.manual_seed(DATA_SEED)

# DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# GENRES_ROOT = f"{DATA_ROOT}/genres_stems"

# GENRES = sorted(os.listdir(GENRES_ROOT))
# STEMS = {"drums.wav":"drums","vocals.wav":"vocals","bass.wav":"bass","other.wav":"other"}
# STEM_KEYS = ['drums','vocals','bass','other']

# GENRE_TO_TEST = 'rock'
# SONG_INDEX = 0


# def build_dataset(root_dir, val_split=0.17, seed=42):
#     train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
#     val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    
#     rng = random.Random(seed)
#     corrupted_count = 0
#     small_count = 0
#     large_count = 0
    
#     for genre in GENRES:
#         genre_path = os.path.join(root_dir,"genres_stems",genre)
#         songs = sorted(os.listdir(genre_path))
#         valid_songs = []
        
#         for song in songs:
#             song_path = os.path.join(genre_path,song)
#             valid=True
            
#             for stem_file in STEMS.keys():
#                 fp = os.path.join(song_path,stem_file)
                
#                 if not os.path.exists(fp):
#                     valid=False
#                 else:
#                     size=os.path.getsize(fp)
                    
#                     if size < 4*1024:
#                         corrupted_count+=1
#                         valid=False
                    
#                     size_mb=size/(1024*1024)
#                     if size_mb < 5.0491: small_count+=1
#                     if size_mb > 5.0493: large_count+=1
            
#             if valid:
#                 valid_songs.append(song)
        
#         rng.shuffle(valid_songs)
#         split=int(len(valid_songs)*(1-val_split))
#         train_songs=valid_songs[:split]
#         val_songs=valid_songs[split:]
        
#         for s in train_songs:
#             for stem_file,stem_key in STEMS.items():
#                 train_dataset[genre][stem_key].append(
#                     os.path.join(genre_path,s,stem_file))
        
#         for s in val_songs:
#             for stem_file,stem_key in STEMS.items():
#                 val_dataset[genre][stem_key].append(
#                     os.path.join(genre_path,s,stem_file))
    
#     Q1 = corrupted_count + small_count
#     Q2 = abs(large_count - small_count)
#     Q3 = abs(len(train_dataset["reggae"]["drums"]) - len(val_dataset["country"]["vocals"]))
    
#     return train_dataset, val_dataset, Q1, Q2, Q3

# tr, val, Q1, Q2, Q3 = build_dataset(DATA_ROOT)

In [ ]:
# def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
#     records = []

#     for genre in dataset_dict:
#         for stem in dataset_dict[genre]:
#             for file_path in dataset_dict[genre][stem]:

#                 y, _ = librosa.load(file_path, sr=sr)

#                 total_duration = len(y) / sr

#                 rms = librosa.feature.rms(
#                     y=y,
#                     frame_length=N_FFT,
#                     hop_length=HOP_LENGTH
#                 )[0]

#                 rms_db = librosa.amplitude_to_db(rms, ref=np.max)

#                 silent_frames = rms_db < -top_db

#                 silence_lengths = []
#                 count = 0

#                 for val in silent_frames:
#                     if val:
#                         count += 1
#                     else:
#                         if count > 0:
#                             silence_lengths.append(count * HOP_LENGTH / sr)
#                             count = 0

#                 if count > 0:
#                     silence_lengths.append(count * HOP_LENGTH / sr)

#                 if len(silence_lengths) == 0:
#                     continue

#                 max_silence = max(silence_lengths)

#                 if max_silence >= threshold_sec:

#                     silence_type = []

#                     if silence_lengths[0] >= threshold_sec:
#                         silence_type.append("start")

#                     if silence_lengths[-1] >= threshold_sec:
#                         silence_type.append("end")

#                     if max_silence >= threshold_sec and not silence_type:
#                         silence_type.append("middle")

#                     records.append({
#                         "Genre": genre,
#                         "Stem": stem,
#                         "Duration": round(total_duration, 2),
#                         "Max_Silence_Sec": round(max_silence, 2),
#                         "Silence_Location": ", ".join(silence_type),
#                         "File_Path": file_path
#                     })

#     columns = ["Genre","Stem","Duration","Max_Silence_Sec","Silence_Location","File_Path"]
#     return pd.DataFrame(records, columns=columns)


In [ ]:
# df_silence = find_long_silences(tr)


# Q4=len(df_silence)
# Q5=len(df_silence[df_silence["Stem"]=="vocals"])
# Q6=df_silence[df_silence["Stem"]=="vocals"]["Max_Silence_Sec"].mean()
# Q7=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums")])
# Q8=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums") & (df_silence["Silence_Location"]=="middle")])
# Q9=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums") & (df_silence["Max_Silence_Sec"]>=10)])


# stems_audio = []
# for key in STEM_KEYS:
#     file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
#     y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
#     stems_audio.append(y)

# stems_stack = np.vstack(stems_audio)

# mix_raw = np.sum(stems_stack, axis=0)

# rms_val = np.sqrt(np.mean(mix_raw ** 2))

# peak_raw = np.max(np.abs(mix_raw))

# mix_norm = mix_raw / peak_raw if peak_raw > 0 else mix_raw

# Q10=len(mix_raw)
# Q11=rms_val
# Q12=np.max(np.abs(mix_raw))


# print("\nFINAL ANSWERS:")
# print("Q1:",Q1)
# print("Q2:",Q2)
# print("Q3:",Q3)
# print("Q4:",Q4)
# print("Q5:",Q5)
# print("Q6:",Q6)
# print("Q7:",Q7)
# print("Q8:",Q8)
# print("Q9:",Q9)
# print("Q10:",Q10)
# print("Q11:",Q11)
# print("Q12:",Q12)

## Milestone 2

In [ ]:
# ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# STEMS_PATH = f"{ROOT}/genres_stems"
# NOISE_PATH = f"{ROOT}/ESC-50-master/audio"
# MASHUPS_PATH = f"{ROOT}/mashups"

# GENRES = ["blues","classical","country","disco","hiphop",
#           "jazz","metal","pop","reggae","rock"]

In [ ]:

# durations=[]

# for song in os.listdir(f"{STEMS_PATH}/jazz"):

#     song_path=f"{STEMS_PATH}/jazz/{song}"

#     for stem in ["drums.wav","vocals.wav","bass.wav","other.wav"]:

#         file=f"{song_path}/{stem}"

#         if os.path.exists(file):

#             y,sr=librosa.load(file,sr=None)

#             durations.append(len(y)/sr)

# print("Q1 Mean Jazz Duration:",np.mean(durations))



# sample_rates=set()

# for root,_,files in os.walk(ROOT):

#     for f in files:

#         if f.endswith(".wav"):

#             file=os.path.join(root,f)

#             try:
#                 _,sr=librosa.load(file,sr=None)
#                 sample_rates.add(sr)
#             except:
#                 pass

# print("Q2 Sample Rates:",sorted(sample_rates))



# empty_files=0

# for root,_,files in os.walk(STEMS_PATH):

#     for f in files:

#         if f.endswith(".wav"):

#             file=os.path.join(root,f)

#             if os.path.getsize(file)==0:

#                 empty_files+=1

# print("Q3 Empty Files:",empty_files)


# db_vals=[]

# for g in GENRES:

#     for song in os.listdir(f"{STEMS_PATH}/{g}"):

#         file=f"{STEMS_PATH}/{g}/{song}/vocals.wav"

#         if os.path.exists(file):

#             y,sr=librosa.load(file,sr=None)

#             peak=np.max(np.abs(y))

#             db=20*np.log10(peak+1e-9)

#             db_vals.append(db)

# print("Q4 Vocal Peak dB:",np.mean(db_vals))


# centroids=[]

# for song in os.listdir(f"{STEMS_PATH}/blues"):

#     file=f"{STEMS_PATH}/blues/{song}/other.wav"

#     if os.path.exists(file):

#         y,sr=librosa.load(file,sr=None)

#         c=np.mean(librosa.feature.spectral_centroid(y=y,sr=sr))

#         centroids.append(c)

# print("Q5 Blues Centroid:",np.mean(centroids))



# genre_centroids={}

# for g in GENRES:

#     vals=[]

#     for song in os.listdir(f"{STEMS_PATH}/{g}"):

#         file=f"{STEMS_PATH}/{g}/{song}/other.wav"

#         if os.path.exists(file):

#             y,sr=librosa.load(file,sr=None)

#             c=np.mean(librosa.feature.spectral_centroid(y=y,sr=sr))

#             vals.append(c)

#     genre_centroids[g]=np.mean(vals)

# print("Q6 Highest Centroid Genre:",
#       max(genre_centroids,key=genre_centroids.get))



# silence_count=0

# for g in GENRES:

#     for song in os.listdir(f"{STEMS_PATH}/{g}"):

#         for stem in ["drums.wav","vocals.wav","bass.wav","other.wav"]:

#             file=f"{STEMS_PATH}/{g}/{song}/{stem}"

#             if os.path.exists(file):

#                 y,sr=librosa.load(file,sr=None,duration=0.5)

#                 if np.max(np.abs(y))<1e-4:

#                     silence_count+=1

# print("Q7 Silence Stems:",silence_count)

In [ ]:

# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import f1_score,confusion_matrix,classification_report

# GENRES = ["blues","classical","country","disco","hiphop",
#           "jazz","metal","pop","reggae","rock"]


# def extract_dt_features(song_path):

#     y,sr=librosa.load(f"{song_path}/other.wav",
#                       sr=22050,
#                       duration=10)

#     tempo,_=librosa.beat.beat_track(y=y,sr=sr)

#     spec_cent=np.mean(
#         librosa.feature.spectral_centroid(y=y,sr=sr)
#     )

#     zcr=np.mean(
#         librosa.feature.zero_crossing_rate(y)
#     )

#     rolloff=np.mean(
#         librosa.feature.spectral_rolloff(y=y,sr=sr)
#     )

#     return [float(tempo),spec_cent,zcr,rolloff]


# data=[]

# for g in GENRES:

#     gp=f"{STEMS_PATH}/{g}"

#     songs=os.listdir(gp)

#     for s in songs[:50]:

#         data.append({
#             "path":f"{gp}/{s}",
#             "genre":g
#         })

# df=pd.DataFrame(data)

# train_df,val_df=train_test_split(
#     df,
#     test_size=0.2,
#     stratify=df['genre'],
#     random_state=42
# )


# X_train=np.array([
#     extract_dt_features(p)
#     for p in train_df['path']
# ])

# y_train=train_df['genre']


# X_val=np.array([
#     extract_dt_features(p)
#     for p in val_df['path']
# ])

# y_val=val_df['genre']


# clf=DecisionTreeClassifier(
#     max_depth=5,
#     random_state=42
# )

# clf.fit(X_train,y_train)

In [ ]:
# y_pred=clf.predict(X_val)
# macro_f1=f1_score(
#     y_val,
#     y_pred,
#     average='macro'
# )
# cm=confusion_matrix(
#     y_val,
#     y_pred,
#     labels=GENRES
# )
# cr=classification_report(
#     y_val,
#     y_pred
# )
# print("Macro F1:",macro_f1)
# print("\nClassification Report\n")
# print(cr)

In [ ]:
# import seaborn as sns

# plt.figure(figsize=(10,8))

# sns.heatmap(
#     cm,
#     annot=True,
#     fmt='d',
#     xticklabels=GENRES,
#     yticklabels=GENRES
# )

# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.title("Confusion Matrix")

# plt.show()

In [ ]:
# from sklearn.metrics import accuracy_score

# accuracy=accuracy_score(y_val,y_pred)

# print("Accuracy:",accuracy)


# tp={}
# fn={}

# total=np.sum(cm)

# for i,g in enumerate(GENRES):

#     TP=cm[i,i]

#     FN=np.sum(cm[i,:])-TP

#     tp[g]=TP
#     fn[g]=FN


# print("Highest TP:",max(tp,key=tp.get))

# print("Lowest FN:",min(fn,key=fn.get))

## Milestone 3

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torchaudio
# from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import train_test_split

# DATA_PATH = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

# GENRES = sorted(os.listdir(DATA_PATH))
# label_map = {g:i for i,g in enumerate(GENRES)}

# mel_transform = torchaudio.transforms.MelSpectrogram(
#     sample_rate=16000,
#     n_fft=400,
#     hop_length=160,
#     n_mels=64
# )

# class AudioDataset(Dataset):
#     def __init__(self, data):
#         self.data = data

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         item = self.data[idx]
#         y, sr = torchaudio.load(item["path"])
#         y = y.mean(dim=0)

#         if y.shape[0] < 16000:
#             y = torch.nn.functional.pad(y, (0, 16000 - y.shape[0]))
#         else:
#             y = y[:16000]

#         mel = mel_transform(y)
#         return mel.unsqueeze(0), label_map[item["genre"]]

# data = []
# for g in GENRES:
#     gpath = os.path.join(DATA_PATH, g)
#     for track in os.listdir(gpath):
#         path = os.path.join(gpath, track, "other.wav")
#         data.append({"path": path, "genre": g})

# train_data, val_data = train_test_split(
#     data,
#     test_size=0.2,
#     stratify=[d["genre"] for d in data],
#     random_state=42
# )

# train_loader = DataLoader(AudioDataset(train_data), batch_size=32, shuffle=True)
# val_loader = DataLoader(AudioDataset(val_data), batch_size=32)

# class CNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv1 = nn.Conv2d(1, 16, 3, 1, 1)
#         self.pool = nn.MaxPool2d(2,2)
#         self.conv2 = nn.Conv2d(16, 32, 3, 1, 1)
#         self.fc1 = nn.Linear(32*16*16, 128)
#         self.fc2 = nn.Linear(128, 10)

#     def forward(self, x):
#         x = self.pool(torch.relu(self.conv1(x)))
#         x = self.pool(torch.relu(self.conv2(x)))
#         x = torch.flatten(x, 1)
#         x = torch.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = CNN().to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.01)

# for epoch in range(5):
#     model.train()
#     total_loss = 0

#     for x, y in train_loader:
#         x, y = x.to(device), y.to(device)

#         optimizer.zero_grad()
#         outputs = model(x)
#         loss = criterion(outputs, y)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# model.eval()
# correct = 0
# total = 0

# with torch.no_grad():
#     for x, y in val_loader:
#         x, y = x.to(device), y.to(device)
#         outputs = model(x)
#         preds = torch.argmax(outputs, dim=1)

#         correct += (preds == y).sum().item()
#         total += y.size(0)

# print("Validation Accuracy:", correct/total)

## Milestone 4

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torchaudio
# import numpy as np
# import random
# import os
# import glob
# from pathlib import Path
# from torch.utils.data import Dataset, DataLoader

# random.seed(42)
# np.random.seed(42)
# torch.manual_seed(42)

# INPUT_BASE="/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
# WORKING_BASE="/kaggle/working"

# STEMS_PATH=f"{INPUT_BASE}/genres_stems"
# NOISE_PATH=f"{INPUT_BASE}/ESC-50-master/audio"

# SYNTH_PATH=f"{WORKING_BASE}/synthetic_mashups/train"
# FEATURE_PATH=f"{WORKING_BASE}/features/train"

# def generate_synthetic_dataset(stems_dir,noise_dir,output_dir,samples_per_genre=50,target_sr=22050,duration=30):

#     genres=["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
#     target_length=target_sr*duration
#     noise_files=glob.glob(os.path.join(noise_dir,"*.wav"))

#     for genre in genres:

#         genre_path=os.path.join(stems_dir,genre)
#         song_folders=[os.path.join(genre_path,d) for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path,d))]

#         genre_out_dir=Path(output_dir)/genre
#         genre_out_dir.mkdir(parents=True,exist_ok=True)

#         for i in range(samples_per_genre):

#             chosen_songs=random.choices(song_folders,k=4)
#             stems=[]
#             stem_types=["drums.wav","vocals.wav","bass.wav","other.wav"]

#             for song,stem_type in zip(chosen_songs,stem_types):

#                 stem_path=os.path.join(song,stem_type)

#                 if not os.path.exists(stem_path):
#                     continue

#                 waveform,sr=torchaudio.load(stem_path)

#                 if sr!=target_sr:
#                     waveform=torchaudio.transforms.Resample(sr,target_sr)(waveform)

#                 if waveform.shape[1]>target_length:
#                     waveform=waveform[:,:target_length]

#                 elif waveform.shape[1]<target_length:
#                     waveform=torch.nn.functional.pad(waveform,(0,target_length-waveform.shape[1]))

#                 stems.append(waveform)

#             if len(stems)==0:
#                 continue

#             mashup=torch.zeros_like(stems[0])

#             for s in stems:
#                 mashup+=s

#             mashup=mashup/(torch.max(torch.abs(mashup))+1e-8)

#             noise_file=random.choice(noise_files)
#             noise,_=torchaudio.load(noise_file)

#             if noise.shape[1]>target_length:
#                 noise=noise[:,:target_length]

#             start_idx=random.randint(0,target_length-noise.shape[1])
#             intensity=random.uniform(0.1,0.4)

#             mashup[:,start_idx:start_idx+noise.shape[1]]+=noise*intensity
#             mashup=mashup/(torch.max(torch.abs(mashup))+1e-8)

#             out_path=genre_out_dir/f"mashup_{i:03d}.wav"
#             torchaudio.save(str(out_path),mashup,target_sr)

# generate_synthetic_dataset(STEMS_PATH,NOISE_PATH,SYNTH_PATH)

# def extract_and_save_features(input_dir,output_dir,target_sr=22050):

#     mel_transform=torchaudio.transforms.MelSpectrogram(sample_rate=target_sr,n_fft=2048,hop_length=512,n_mels=128)
#     amplitude_to_db=torchaudio.transforms.AmplitudeToDB()

#     wav_files=glob.glob(os.path.join(input_dir,"**","*.wav"),recursive=True)

#     for wav_path in wav_files:

#         rel_path=os.path.relpath(wav_path,input_dir)
#         out_path=Path(output_dir)/rel_path
#         out_path=out_path.with_suffix(".pt")

#         out_path.parent.mkdir(parents=True,exist_ok=True)

#         waveform,sr=torchaudio.load(wav_path)

#         waveform=waveform.mean(dim=0,keepdim=True)

#         mel_spec=mel_transform(waveform)
#         mel_spec_db=amplitude_to_db(mel_spec)

#         torch.save(mel_spec_db,out_path)

#     print("Saved",len(wav_files),"feature files")

# extract_and_save_features(SYNTH_PATH,FEATURE_PATH)

# class PrecomputedFeatureDataset(Dataset):

#     def __init__(self,features_dir):

#         self.files=glob.glob(os.path.join(features_dir,"**","*.pt"),recursive=True)

#         self.genres=sorted(["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"])
#         self.genre_to_idx={g:i for i,g in enumerate(self.genres)}

#     def __len__(self):
#         return len(self.files)

#     def __getitem__(self,idx):

#         file_path=self.files[idx]
#         genre=Path(file_path).parent.name
#         label=self.genre_to_idx[genre]

#         feature=torch.load(file_path)

#         return feature,label

# class CRNN(nn.Module):

#     def __init__(self,num_classes=10):

#         super().__init__()

#         self.cnn=nn.Sequential(

#             nn.Conv2d(1,32,3,padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),

#             nn.Conv2d(32,64,3,padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2)
#         )

#         self.lstm=nn.LSTM(input_size=2048,hidden_size=64,batch_first=True,bidirectional=True)

#         self.fc=nn.Linear(128,num_classes)

#     def forward(self,x):

#         x=self.cnn(x)

#         b,c,f,t=x.shape

#         x=x.permute(0,3,1,2).reshape(b,t,c*f)

#         lstm_out,_=self.lstm(x)

#         pooled,_=torch.max(lstm_out,dim=1)

#         logits=self.fc(pooled)

#         return logits

# dataset=PrecomputedFeatureDataset(FEATURE_PATH)

# train_size=int(0.8*len(dataset))
# val_size=len(dataset)-train_size

# train_dataset,val_dataset=torch.utils.data.random_split(dataset,[train_size,val_size])

# train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
# val_loader=DataLoader(val_dataset,batch_size=32)

# device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model=CRNN(num_classes=10).to(device)

# criterion=nn.CrossEntropyLoss()
# optimizer=optim.Adam(model.parameters(),lr=0.001)

# for epoch in range(10):

#     model.train()

#     for x,y in train_loader:

#         x=x.to(device)
#         y=y.to(device)

#         optimizer.zero_grad()

#         outputs=model(x)

#         loss=criterion(outputs,y)

#         loss.backward()

#         optimizer.step()

# wav_files=len(glob.glob("/kaggle/working/synthetic_mashups/train/**/*.wav",recursive=True))

# waveform,_=torchaudio.load(glob.glob("/kaggle/working/synthetic_mashups/train/**/*.wav",recursive=True)[0])

# feature=torch.load(glob.glob("/kaggle/working/features/train/**/*.pt",recursive=True)[0])

# lstm_params=sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

# wav_files = glob.glob("/kaggle/working/synthetic_mashups/train/**/*.wav", recursive=True)
# print("Q1:", len(wav_files))

# waveform, _ = torchaudio.load(wav_files[0])
# print("Q2:", tuple(waveform.shape))

# feature_files = glob.glob("/kaggle/working/features/train/**/*.pt", recursive=True)
# feature = torch.load(feature_files[0])
# print("Q3:", tuple(feature.shape))

# x = feature.unsqueeze(0).to(device)
# with torch.no_grad():
#     cnn_out = model.cnn(x)
# print("Q4:", tuple(cnn_out.shape))

# lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
# print("Q5:", lstm_params)


## Milestone 5

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import librosa
# from sklearn.model_selection import train_test_split
# from transformers import AutoFeatureExtractor, ASTForAudioClassification

# DATA_PATH = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
# GENRES_PATH = f"{DATA_PATH}/genres_stems"
# ESC_PATH = f"{DATA_PATH}/ESC-50-master/audio"

# STEMS = ["drums.wav", "vocals.wav", "bass.wav", "other.wav"]

# GENRES = sorted(os.listdir(GENRES_PATH))

# recipes = []
# for genre in GENRES:
#     genre_path = os.path.join(GENRES_PATH, genre)
#     for track in os.listdir(genre_path):
#         recipes.append({"genre": genre, "track_path": os.path.join(genre_path, track)})

# train_recipes, val_recipes = train_test_split(
#     recipes,
#     test_size=0.2,
#     shuffle=True,
#     random_state=42
# )

# print(len(val_recipes))

# sample_recipe = train_recipes[0]
# track_path = sample_recipe["track_path"]

# loaded_stems = []
# for stem in STEMS:
#     y, _ = librosa.load(os.path.join(track_path, stem), sr=16000, duration=10)
#     if len(y) < 160000:
#         y = np.pad(y, (0, 160000 - len(y)))
#     else:
#         y = y[:160000]
#     loaded_stems.append(y)

# noise_file = os.listdir(ESC_PATH)[0]
# noise_path = os.path.join(ESC_PATH, noise_file)

# noise, _ = librosa.load(noise_path, sr=16000, duration=10)
# if len(noise) < 160000:
#     noise = np.pad(noise, (0, 160000 - len(noise)))
# else:
#     noise = noise[:160000]

# mix = sum(loaded_stems)
# final_mix = mix + 0.2 * noise

# print(final_mix.shape)

# extractor = AutoFeatureExtractor.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593"
# )

# inputs = extractor(
#     final_mix,
#     sampling_rate=16000,
#     return_tensors="pt"
# )

# input_values = inputs["input_values"].squeeze(0)
# print(tuple(input_values.shape))

# model = ASTForAudioClassification.from_pretrained(
#     "MIT/ast-finetuned-audioset-10-10-0.4593",
#     num_labels=10,
#     ignore_mismatched_sizes=True
# )

# params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(params)

# y_test = np.array([-0.85, 0.40, 0.20, -0.10])
# y = y_test / (np.max(np.abs(y_test)) + 1e-9)
# print(round(y[0], 3))